<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Event Sequence Custom Embedding Transformer
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>
<p style="font-size:16px;font-family:Arial"> In this notebook we will train a <b>vector embedding model</b> for event sequences. Unlike the predictive model, this model's primary purpose is to generate <b>fixed-size embeddings</b> for sequences that can be used for:
 <ul style="font-size:16px;font-family:Arial">   
     <li><b>Similarity search</b>: Find similar customer journeys</li>
<li><b>Clustering</b>: Group customers by behavior patterns</li>
<li><b>Classification</b>: Zero-shot or few-shot classification</li>
<li><b>Anomaly detection</b>: Find unusual sequences</li>
<li><b>Retrieval</b>: Power RAG systems with sequence context</li>
     </ul>
<p style="font-size:18px;font-family:Arial"> <b>Output</b></p>
<ul style="font-size:16px;font-family:Arial">
The model will be saved in HuggingFace-compatible format with:
<li><code>pytorch_model.bin</code> - Model weights</li>
<li><code>config.json</code> - Model configuration</li>
<li><code>tokenizer.json</code> - HuggingFace tokenizer format</li>
<li><code>vocab.json</code> - Event vocabulary</li>
<li><code>tokenizer_config.json</code> - Tokenizer settings</li>
<li><code>special_tokens_map.json</code> - Special token definitions</li>
   </ul>
<p style="font-size:18px;font-family:Arial">The overall processing pipeline follows the sequence as below:</p> 
<img src="./images/transformer.png" alt="transformer" style="width:100%; border: 4px solid #404040; border-radius: 10px;" />
<br>
<p style="font-size:18px;font-family:Arial"><i>*This notebook takes approx 15min to run</i></p> 

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Configuration & Hyperparameters</b></p>
<p style = 'font-size:16px;font-family:Arial'>All tunable parameters are defined here for easy experimentation.</p>

In [ ]:
# =============================================================================
# CONFIGURATION - MODIFY THESE PARAMETERS
# =============================================================================
EVENT_COLUMN = "Event"                  # Column name for events
USER_COLUMN = "UserID"
SESSION_COLUMN = "SessionID"            # Column name for session IDs
TIMESTAMP_COLUMN = "Event_TS"          # Column name for timestamps

In [ ]:
# ----- Model Architecture -----
# Choose a preset or customize below
MODEL_SIZE = "small"  # Options: "tiny", "small", "medium", "large", "custom"

# Custom architecture (used if MODEL_SIZE = "custom")
CUSTOM_CONFIG = {
    "embedding_dim": 256,      # Output embedding dimension (vector length)
    "d_model": 256,            # Internal model dimension
    "num_heads": 8,            # Number of attention heads
    "num_layers": 4,           # Number of transformer layers
    "d_ff": 1024,              # Feed-forward hidden dimension
    "dropout": 0.1,            # Dropout rate
}

# Preset configurations
MODEL_PRESETS = {
    "tiny": {
        "embedding_dim": 128,
        "d_model": 128,
        "num_heads": 4,
        "num_layers": 2,
        "d_ff": 512,
        "dropout": 0.1,
    },
    "small": {
        "embedding_dim": 256,
        "d_model": 256,
        "num_heads": 8,
        "num_layers": 4,
        "d_ff": 1024,
        "dropout": 0.1,
    },
    "medium": {
        "embedding_dim": 512,
        "d_model": 512,
        "num_heads": 8,
        "num_layers": 6,
        "d_ff": 2048,
        "dropout": 0.1,
    },
    "large": {
        "embedding_dim": 768,
        "d_model": 768,
        "num_heads": 12,
        "num_layers": 8,
        "d_ff": 3072,
        "dropout": 0.1,
    },
}

# ----- Training Hyperparameters -----
MAX_SEQ_LEN = 64               # Maximum sequence length
BATCH_SIZE = 32                # Batch size
NUM_EPOCHS = 10                # Number of training epochs
LEARNING_RATE = 3e-4           # Learning rate
WEIGHT_DECAY = 0.01            # L2 regularization
WARMUP_STEPS = 100             # Learning rate warmup steps
MAX_GRAD_NORM = 1.0            # Gradient clipping
EARLY_STOPPING_PATIENCE = 3   # Stop if no improvement for N epochs

# ----- Contrastive Learning Parameters -----
TEMPERATURE = 0.07             # Temperature for contrastive loss (lower = harder negatives)
POOLING_STRATEGY = "mean"      # Options: "mean", "max", "cls", "last"
NORMALIZE_EMBEDDINGS = True    # L2 normalize output embeddings

# ----- Data Augmentation -----
AUGMENT_DROP_PROB = 0.1        # Probability of dropping an event
AUGMENT_SHUFFLE_PROB = 0.0     # Probability of shuffling adjacent events (use carefully)

# ----- Data Split -----
TEST_SIZE = 0.1                # Fraction for test set
VAL_SIZE = 0.1                 # Fraction for validation set
RANDOM_SEED = 42               # For reproducibility

# ----- Output -----
OUTPUT_DIR = "./embedding_model"  # Where to save the trained model

print("Configuration loaded!")
print(f"Model size: {MODEL_SIZE}")
print(f"Output embedding dimension: {MODEL_PRESETS.get(MODEL_SIZE, CUSTOM_CONFIG)['embedding_dim']}")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>2. Setup and Imports </b></p>

In [ ]:
import os
import json
import math
import random
from typing import List, Dict, Optional, Tuple, Any
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

from teradataml import *

In [ ]:
# Device setup with Apple Silicon (MPS) support
def get_device():
    """
    Get the best available device:
    1. CUDA (NVIDIA GPU)
    2. MPS (Apple Silicon - M1/M2/M3)
    3. CPU (fallback)
    """
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = get_device()
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
elif DEVICE.type == 'mps':
    print("  Apple Silicon GPU (MPS) enabled")

In [ ]:
# Set random seeds for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
print(f"Random seed set to: {RANDOM_SEED}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=4._Bank_Clickstream_-_Custom_Embedding_Transformer.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>4. Load and Explore Data</b></p>

In [ ]:
# Load your data
#df = pd.read_csv(DATA_PATH)
df = DataFrame(in_schema('DEMO_Bank','Session_Events'))
print(f"Data shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head()

In [ ]:
df=df.to_pandas().reset_index()

In [ ]:
# Data summary
print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)
print(f"Total events: {len(df):,}")
print(f"Unique sessions: {df[[USER_COLUMN, SESSION_COLUMN]].apply(tuple, axis=1).unique()}")
print(f"Unique event types: {df[EVENT_COLUMN].nunique():,}")

print(f"\nTop 15 Events:")
print(df[EVENT_COLUMN].value_counts().head(15))

In [ ]:
# Create sequences from data
df_sorted = df.sort_values([USER_COLUMN, SESSION_COLUMN, TIMESTAMP_COLUMN])
session_sequences = df_sorted.groupby([USER_COLUMN, SESSION_COLUMN])[EVENT_COLUMN].apply(list).tolist()

print(f"Total sequences: {len(session_sequences):,}")
print(f"Average sequence length: {np.mean([len(s) for s in session_sequences]):.1f}")
print(f"Min length: {min(len(s) for s in session_sequences)}")
print(f"Max length: {max(len(s) for s in session_sequences)}")

# Distribution of sequence lengths
lengths = [len(s) for s in session_sequences]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Sequence Length')
plt.ylabel('Count')
plt.title('Distribution of Sequence Lengths')
plt.axvline(x=MAX_SEQ_LEN, color='r', linestyle='--', label=f'Max length ({MAX_SEQ_LEN})')
plt.legend()
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>5. Tokenizer</b></p>
<p style = 'font-size:16px;font-family:Arial'>Build a tokenizer that converts events to token IDs. Saved in HuggingFace-compatible format.</p>

In [ ]:
class EventTokenizer:
    """
    Tokenizer for event sequences with HuggingFace-compatible export.
    
    Special tokens:
        [PAD] (0): Padding
        [UNK] (1): Unknown events
        [CLS] (2): Classification token (sequence start)
        [SEP] (3): Separator / End of sequence
        [MASK] (4): Mask token for MLM
    """
    
    SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
    
    def __init__(self, vocab: Optional[Dict[str, int]] = None, min_freq: int = 1):
        self.min_freq = min_freq
        
        if vocab is not None:
            self.vocab = vocab
            self.id_to_token = {v: k for k, v in vocab.items()}
        else:
            self.vocab = {tok: i for i, tok in enumerate(self.SPECIAL_TOKENS)}
            self.id_to_token = {i: tok for i, tok in enumerate(self.SPECIAL_TOKENS)}
    
    @property
    def pad_token_id(self) -> int:
        return 0
    
    @property
    def unk_token_id(self) -> int:
        return 1
    
    @property
    def cls_token_id(self) -> int:
        return 2
    
    @property
    def sep_token_id(self) -> int:
        return 3
    
    @property
    def mask_token_id(self) -> int:
        return 4
    
    @property
    def vocab_size(self) -> int:
        return len(self.vocab)
    
    def fit(self, sequences: List[List[str]]) -> "EventTokenizer":
        """Build vocabulary from sequences."""
        event_counts = Counter()
        for seq in sequences:
            event_counts.update(seq)
        
        next_id = len(self.SPECIAL_TOKENS)
        for event, count in event_counts.most_common():
            if count >= self.min_freq and event not in self.vocab:
                self.vocab[event] = next_id
                self.id_to_token[next_id] = event
                next_id += 1
        
        return self
    
    def encode(
        self,
        sequence: List[str],
        add_special_tokens: bool = True,
        max_length: Optional[int] = None,
        padding: bool = False,
        truncation: bool = False
    ) -> Dict[str, List[int]]:
        """Encode sequence to token IDs with attention mask."""
        token_ids = [self.vocab.get(event, self.unk_token_id) for event in sequence]
        
        if add_special_tokens:
            token_ids = [self.cls_token_id] + token_ids + [self.sep_token_id]
        
        if truncation and max_length:
            token_ids = token_ids[:max_length]
        
        seq_len = len(token_ids)
        attention_mask = [1] * seq_len
        
        if padding and max_length:
            pad_len = max_length - seq_len
            if pad_len > 0:
                token_ids = token_ids + [self.pad_token_id] * pad_len
                attention_mask = attention_mask + [0] * pad_len
        
        return {
            "input_ids": token_ids,
            "attention_mask": attention_mask
        }
    
    def decode(self, token_ids: List[int], skip_special_tokens: bool = True) -> List[str]:
        """Decode token IDs to events."""
        events = []
        for tid in token_ids:
            token = self.id_to_token.get(tid, "[UNK]")
            if skip_special_tokens and token in self.SPECIAL_TOKENS:
                continue
            events.append(token)
        return events
    
    def save(self, path: str):
        """Save tokenizer in HuggingFace-compatible format."""
        os.makedirs(path, exist_ok=True)
        
        # vocab.json
        with open(os.path.join(path, "vocab.json"), "w") as f:
            json.dump(self.vocab, f, indent=2)
        
        # tokenizer.json (HuggingFace format)
        tokenizer_json = {
            "version": "1.0",
            "truncation": None,
            "padding": None,
            "added_tokens": [
                {
                    "id": self.vocab[tok],
                    "content": tok,
                    "single_word": False,
                    "lstrip": False,
                    "rstrip": False,
                    "normalized": False,
                    "special": True
                }
                for tok in self.SPECIAL_TOKENS
            ],
            "normalizer": None,
            "pre_tokenizer": {"type": "WhitespaceSplit"},
            "post_processor": {
                "type": "TemplateProcessing",
                "single": [
                    {"SpecialToken": {"id": "[CLS]", "type_id": 0}},
                    {"Sequence": {"id": "A", "type_id": 0}},
                    {"SpecialToken": {"id": "[SEP]", "type_id": 0}}
                ],
                "pair": None,
                "special_tokens": {
                    "[CLS]": {"id": "[CLS]", "ids": [self.cls_token_id], "tokens": ["[CLS]"]},
                    "[SEP]": {"id": "[SEP]", "ids": [self.sep_token_id], "tokens": ["[SEP]"]}
                }
            },
            "decoder": None,
            "model": {
                "type": "WordLevel",
                "vocab": self.vocab,
                "unk_token": "[UNK]"
            }
        }
        with open(os.path.join(path, "tokenizer.json"), "w") as f:
            json.dump(tokenizer_json, f, indent=2)
        
        # tokenizer_config.json
        config = {
            "tokenizer_class": "PreTrainedTokenizerFast",
            "cls_token": "[CLS]",
            "sep_token": "[SEP]",
            "unk_token": "[UNK]",
            "pad_token": "[PAD]",
            "mask_token": "[MASK]",
            "model_max_length": 512,
            "padding_side": "right",
            "truncation_side": "right"
        }
        with open(os.path.join(path, "tokenizer_config.json"), "w") as f:
            json.dump(config, f, indent=2)
        
        # special_tokens_map.json
        special_map = {
            "cls_token": "[CLS]",
            "sep_token": "[SEP]",
            "unk_token": "[UNK]",
            "pad_token": "[PAD]",
            "mask_token": "[MASK]"
        }
        with open(os.path.join(path, "special_tokens_map.json"), "w") as f:
            json.dump(special_map, f, indent=2)
    
    @classmethod
    def load(cls, path: str) -> "EventTokenizer":
        """Load tokenizer from directory."""
        with open(os.path.join(path, "vocab.json"), "r") as f:
            vocab = json.load(f)
        return cls(vocab=vocab)

In [ ]:
# Build tokenizer
tokenizer = EventTokenizer(min_freq=1)
tokenizer.fit(session_sequences)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: {EventTokenizer.SPECIAL_TOKENS}")
print(f"\nSample vocabulary (first 20):")
for event, idx in list(tokenizer.vocab.items())[:20]:
    print(f"  {idx:3d}: {event}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>6. Dataset with Contrastive Learning</b></p>
<ul style = 'font-size:16px;font-family:Arial'>
    For embedding models, we use <b>contrastive learning</b>:
    <li>Positive pairs: Augmented versions of the same sequence should be similar</li>
    <li>Negative pairs: Different sequences should be dissimilar</li>
    </ul>
<p style = 'font-size:16px;font-family:Arial'>    
We use <b>SimCLR-style</b> training where each sequence generates two augmented views.

In [ ]:
class SequenceAugmenter:
    """
    Data augmentation for event sequences.
    Creates positive pairs for contrastive learning.
    """
    
    def __init__(
        self,
        drop_prob: float = 0.1,
        shuffle_prob: float = 0.0,
        min_length: int = 2
    ):
        self.drop_prob = drop_prob
        self.shuffle_prob = shuffle_prob
        self.min_length = min_length
    
    def augment(self, sequence: List[str]) -> List[str]:
        """Apply random augmentations to a sequence."""
        augmented = sequence.copy()
        
        # Random drop
        if self.drop_prob > 0 and len(augmented) > self.min_length:
            augmented = [
                event for event in augmented
                if random.random() > self.drop_prob
            ]
            # Ensure minimum length
            if len(augmented) < self.min_length:
                augmented = sequence[:self.min_length]
        
        # Random adjacent shuffle (use carefully - may break temporal patterns)
        if self.shuffle_prob > 0 and len(augmented) > 2:
            for i in range(len(augmented) - 1):
                if random.random() < self.shuffle_prob:
                    augmented[i], augmented[i+1] = augmented[i+1], augmented[i]
        
        return augmented if augmented else sequence

In [ ]:
class ContrastiveDataset(Dataset):
    """
    Dataset for contrastive learning.
    Each item returns two augmented views of the same sequence.
    """
    
    def __init__(
        self,
        sequences: List[List[str]],
        tokenizer: EventTokenizer,
        max_length: int = 64,
        augmenter: Optional[SequenceAugmenter] = None
    ):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augmenter = augmenter or SequenceAugmenter()
    
    def __len__(self) -> int:
        return len(self.sequences)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sequence = self.sequences[idx]
        
        # Create two augmented views
        view1 = self.augmenter.augment(sequence)
        view2 = self.augmenter.augment(sequence)
        
        # Encode both views
        encoded1 = self.tokenizer.encode(
            view1,
            add_special_tokens=True,
            max_length=self.max_length,
            padding=True,
            truncation=True
        )
        encoded2 = self.tokenizer.encode(
            view2,
            add_special_tokens=True,
            max_length=self.max_length,
            padding=True,
            truncation=True
        )
        
        return {
            "input_ids_1": torch.tensor(encoded1["input_ids"], dtype=torch.long),
            "attention_mask_1": torch.tensor(encoded1["attention_mask"], dtype=torch.long),
            "input_ids_2": torch.tensor(encoded2["input_ids"], dtype=torch.long),
            "attention_mask_2": torch.tensor(encoded2["attention_mask"], dtype=torch.long),
        }


class SimpleDataset(Dataset):
    """
    Simple dataset for inference/evaluation (no augmentation).
    """
    
    def __init__(
        self,
        sequences: List[List[str]],
        tokenizer: EventTokenizer,
        max_length: int = 64
    ):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self) -> int:
        return len(self.sequences)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoded = self.tokenizer.encode(
            self.sequences[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding=True,
            truncation=True
        )
        
        return {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
        }

In [ ]:
# Filter short sequences
min_seq_len = 3
filtered_sequences = [seq for seq in session_sequences if len(seq) >= min_seq_len]
print(f"Sequences after filtering (min length {min_seq_len}): {len(filtered_sequences):,}")

# Split data
indices = np.random.permutation(len(filtered_sequences))
n_test = int(len(filtered_sequences) * TEST_SIZE)
n_val = int(len(filtered_sequences) * VAL_SIZE)

test_indices = indices[:n_test]
val_indices = indices[n_test:n_test + n_val]
train_indices = indices[n_test + n_val:]

train_seqs = [filtered_sequences[i] for i in train_indices]
val_seqs = [filtered_sequences[i] for i in val_indices]
test_seqs = [filtered_sequences[i] for i in test_indices]

print(f"Train: {len(train_seqs):,}, Val: {len(val_seqs):,}, Test: {len(test_seqs):,}")

In [ ]:
# Create datasets and dataloaders
augmenter = SequenceAugmenter(
    drop_prob=AUGMENT_DROP_PROB,
    shuffle_prob=AUGMENT_SHUFFLE_PROB
)

train_dataset = ContrastiveDataset(train_seqs, tokenizer, MAX_SEQ_LEN, augmenter)
val_dataset = ContrastiveDataset(val_seqs, tokenizer, MAX_SEQ_LEN, augmenter)
test_dataset = SimpleDataset(test_seqs, tokenizer, MAX_SEQ_LEN)

# DataLoader settings
loader_kwargs = {
    'num_workers': 0,
    'pin_memory': DEVICE.type == 'cuda',
}

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

# Check sample batch
sample = next(iter(train_loader))
print("Sample batch shapes:")
for k, v in sample.items():
    print(f"  {k}: {v.shape}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>7. Embedding Model Architecture</b></p>
<p style = 'font-size:16px;font-family:Arial'>A transformer encoder that produces fixed-size embeddings for sequences.</p>

In [ ]:
class PositionalEncoding(nn.Module):
    """Learned positional embeddings."""
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pe = nn.Embedding(max_len, d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = x + self.pe(positions)
        return self.dropout(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention (bidirectional for encoder)."""
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
    
    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        
        context = torch.matmul(attention, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(context)

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Single transformer encoder block."""
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Pre-LN architecture
        normed = self.norm1(x)
        x = x + self.dropout1(self.attention(normed, mask))
        
        normed = self.norm2(x)
        x = x + self.dropout2(self.feed_forward(normed))
        
        return x

In [ ]:
class EventSequenceEmbedder(nn.Module):
    """
    Transformer encoder for generating sequence embeddings.
    
    Architecture:
    1. Token embedding + positional encoding
    2. Transformer encoder layers (bidirectional attention)
    3. Pooling to fixed-size vector
    4. Optional projection to embedding dimension
    """
    
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 256,
        d_model: int = 256,
        num_heads: int = 8,
        num_layers: int = 4,
        d_ff: int = 1024,
        max_seq_len: int = 512,
        dropout: float = 0.1,
        pad_token_id: int = 0,
        pooling: str = "mean",
        normalize: bool = True
    ):
        """
        Args:
            vocab_size: Number of unique events
            embedding_dim: Output embedding dimension
            d_model: Internal model dimension
            num_heads: Number of attention heads
            num_layers: Number of encoder layers
            d_ff: Feed-forward hidden dimension
            max_seq_len: Maximum sequence length
            dropout: Dropout rate
            pad_token_id: Padding token ID
            pooling: Pooling strategy ("mean", "max", "cls", "last")
            normalize: L2 normalize output embeddings
        """
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.d_ff = d_ff
        self.max_seq_len = max_seq_len
        self.pad_token_id = pad_token_id
        self.pooling = pooling
        self.normalize = normalize
        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Transformer encoder layers
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Final layer norm
        self.final_norm = nn.LayerNorm(d_model)
        
        # Projection to embedding dimension (if different from d_model)
        if embedding_dim != d_model:
            self.projection = nn.Linear(d_model, embedding_dim)
        else:
            self.projection = nn.Identity()
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)
                if module.padding_idx is not None:
                    nn.init.zeros_(module.weight[module.padding_idx])
    
    def _pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Pool sequence representations to a single vector.
        
        Args:
            x: Hidden states (batch, seq_len, d_model)
            attention_mask: Mask (batch, seq_len)
        """
        if self.pooling == "cls":
            # Use [CLS] token (first token)
            return x[:, 0, :]
        
        elif self.pooling == "last":
            # Use last non-padded token
            seq_lengths = attention_mask.sum(dim=1) - 1
            batch_size = x.size(0)
            return x[torch.arange(batch_size, device=x.device), seq_lengths]
        
        elif self.pooling == "mean":
            # Mean pooling over non-padded tokens
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_embeddings = (x * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            return sum_embeddings / sum_mask
        
        elif self.pooling == "max":
            # Max pooling over non-padded tokens
            mask_expanded = attention_mask.unsqueeze(-1).float()
            x_masked = x.masked_fill(mask_expanded == 0, float('-inf'))
            return x_masked.max(dim=1).values
        
        else:
            raise ValueError(f"Unknown pooling strategy: {self.pooling}")
    
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Generate embeddings for input sequences.
        
        Args:
            input_ids: Token IDs (batch, seq_len)
            attention_mask: Attention mask (batch, seq_len)
            
        Returns:
            Embeddings of shape (batch, embedding_dim)
        """
        if attention_mask is None:
            attention_mask = (input_ids != self.pad_token_id).long()
        
        # Token embeddings + positional encoding
        x = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Transformer encoder layers
        for layer in self.layers:
            x = layer(x, attention_mask)
        
        # Final norm
        x = self.final_norm(x)
        
        # Pool to single vector
        pooled = self._pool(x, attention_mask)
        
        # Project to embedding dimension
        embeddings = self.projection(pooled)
        
        # L2 normalize
        if self.normalize:
            embeddings = F.normalize(embeddings, p=2, dim=-1)
        
        return embeddings
    
    def get_config(self) -> Dict[str, Any]:
        """Return model configuration."""
        return {
            "architecture": "EventSequenceEmbedder",
            "vocab_size": self.vocab_size,
            "embedding_dim": self.embedding_dim,
            "d_model": self.d_model,
            "num_heads": self.num_heads,
            "num_layers": self.num_layers,
            "d_ff": self.d_ff,
            "max_seq_len": self.max_seq_len,
            "pad_token_id": self.pad_token_id,
            "pooling": self.pooling,
            "normalize": self.normalize
        }

In [ ]:
# Get model configuration
if MODEL_SIZE == "custom":
    model_config = CUSTOM_CONFIG
else:
    model_config = MODEL_PRESETS[MODEL_SIZE]

print(f"Model configuration ({MODEL_SIZE}):")
for k, v in model_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Create model
model = EventSequenceEmbedder(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=model_config["embedding_dim"],
    d_model=model_config["d_model"],
    num_heads=model_config["num_heads"],
    num_layers=model_config["num_layers"],
    d_ff=model_config["d_ff"],
    max_seq_len=MAX_SEQ_LEN,
    dropout=model_config["dropout"],
    pad_token_id=tokenizer.pad_token_id,
    pooling=POOLING_STRATEGY,
    normalize=NORMALIZE_EMBEDDINGS
)

model = model.to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Summary:")
print(f"  Total parameters: {num_params:,}")
print(f"  Output embedding dimension: {model_config['embedding_dim']}")
print(f"  Pooling strategy: {POOLING_STRATEGY}")
print(f"  Normalize embeddings: {NORMALIZE_EMBEDDINGS}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>8. Contrastive Loss</b></p>
<p style = 'font-size:16px;font-family:Arial'>We use **InfoNCE loss** (also known as NT-Xent from SimCLR):
<ul style = 'font-size:16px;font-family:Arial'>
    <li>Positive pairs (augmented views of same sequence) should be similar</li>
    <li>Negative pairs (different sequences) should be dissimilar</li></ul>

In [ ]:
class InfoNCELoss(nn.Module):
    """
    InfoNCE / NT-Xent contrastive loss.
    
    Given embeddings for two views of N sequences:
    - z1: (N, embedding_dim) - first view
    - z2: (N, embedding_dim) - second view
    
    For each z1[i], the positive is z2[i], and negatives are all other z2[j] (j != i)
    """
    
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        Compute contrastive loss.
        
        Args:
            z1: Embeddings from first view (batch_size, embedding_dim)
            z2: Embeddings from second view (batch_size, embedding_dim)
        """
        batch_size = z1.size(0)
        
        # Normalize embeddings (should already be normalized, but ensure)
        z1 = F.normalize(z1, p=2, dim=1)
        z2 = F.normalize(z2, p=2, dim=1)
        
        # Compute similarity matrix
        # sim[i, j] = cosine_similarity(z1[i], z2[j])
        sim_matrix = torch.matmul(z1, z2.T) / self.temperature  # (batch, batch)
        
        # Labels: positive pairs are on the diagonal
        labels = torch.arange(batch_size, device=z1.device)
        
        # Cross-entropy loss (treating as classification problem)
        # For each z1[i], classify which z2[j] is the positive
        loss = F.cross_entropy(sim_matrix, labels)
        
        return loss

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>9. Training Loop</b></p>

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device, max_grad_norm):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        # Move to device
        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)
        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)
        
        # Forward pass for both views
        z1 = model(input_ids_1, attention_mask_1)
        z2 = model(input_ids_2, attention_mask_2)
        
        # Compute loss
        loss = criterion(z1, z2)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)
        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)
        
        z1 = model(input_ids_1, attention_mask_1)
        z2 = model(input_ids_2, attention_mask_2)
        
        loss = criterion(z1, z2)
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

In [ ]:
# Setup training
criterion = InfoNCELoss(temperature=TEMPERATURE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * NUM_EPOCHS

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=1e-10,
    end_factor=1.0,
    total_iters=WARMUP_STEPS
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps - WARMUP_STEPS,
    eta_min=LEARNING_RATE * 0.1
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_STEPS]
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"Temperature: {TEMPERATURE}")

In [ ]:
# Training loop
print("=" * 60)
print("TRAINING EMBEDDING MODEL")
print("=" * 60)

train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler, 
        criterion, DEVICE, MAX_GRAD_NORM
    )
    train_losses.append(train_loss)
    
    # Validate
    val_loss = evaluate(model, val_loader, criterion, DEVICE)
    val_losses.append(val_loss)
    
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    
    # Check for best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print("  ✓ New best model!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
    
    # Early stopping
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping after {epoch + 1} epochs")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model = model.to(DEVICE)
    print("\n✓ Restored best model")

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Contrastive Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>10. Save Model (HuggingFace Format)</b></p>

In [ ]:
def save_model(model, tokenizer, output_dir: str, training_config: Dict = None):
    """Save model in HuggingFace-compatible format."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save model weights
    torch.save(model.state_dict(), os.path.join(output_dir, "pytorch_model.bin"))
    
    # Save model config
    config = model.get_config()
    config["model_type"] = "event-sequence-embedder"
    config["architectures"] = ["EventSequenceEmbedder"]
    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump(config, f, indent=2)
    
    # Save tokenizer
    tokenizer.save(output_dir)
    
    # Save training config if provided
    if training_config:
        with open(os.path.join(output_dir, "training_config.json"), "w") as f:
            json.dump(training_config, f, indent=2)
    
    # Save README
    readme = f"""# Event Sequence Embedding Model

This model generates vector embeddings for event sequences.

## Model Details

- **Embedding Dimension**: {config['embedding_dim']}
- **Model Dimension**: {config['d_model']}
- **Attention Heads**: {config['num_heads']}
- **Layers**: {config['num_layers']}
- **Vocabulary Size**: {config['vocab_size']}
- **Pooling**: {config['pooling']}
- **Normalized**: {config['normalize']}

## Usage

```python
from model import EventSequenceEmbedder
from tokenizer import EventTokenizer

# Load
tokenizer = EventTokenizer.load("{output_dir}")
model = EventSequenceEmbedder(**config)
model.load_state_dict(torch.load("{output_dir}/pytorch_model.bin"))

# Encode
events = ["Login", "Check Balance", "Transfer"]
encoded = tokenizer.encode(events, add_special_tokens=True, max_length=64, padding=True, truncation=True)
input_ids = torch.tensor([encoded["input_ids"]])
attention_mask = torch.tensor([encoded["attention_mask"]])

# Get embedding
embedding = model(input_ids, attention_mask)  # Shape: (1, {config['embedding_dim']})
```

## Training

Trained using contrastive learning (InfoNCE loss) on event sequences.
"""
    with open(os.path.join(output_dir, "README.md"), "w") as f:
        f.write(readme)
    
    print(f"Model saved to: {output_dir}")
    print("\nSaved files:")
    for f in sorted(os.listdir(output_dir)):
        print(f"  - {f}")

In [ ]:
from datetime import datetime

In [ ]:
# Save the model
training_config = {
    "model_size": MODEL_SIZE,
    "max_seq_len": MAX_SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "temperature": TEMPERATURE,
    "pooling_strategy": POOLING_STRATEGY,
    "augment_drop_prob": AUGMENT_DROP_PROB,
    "train_sequences": len(train_seqs),
    "val_sequences": len(val_seqs),
    "test_sequences": len(test_seqs),
    "final_train_loss": train_losses[-1],
    "final_val_loss": val_losses[-1],
    "best_val_loss": best_val_loss,
    "trained_at": datetime.now().isoformat()
}

save_model(model, tokenizer, OUTPUT_DIR, training_config)

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>11. Test Embeddings</b></p>

In [ ]:
@torch.no_grad()
def get_embedding(model, tokenizer, events: List[str], device=DEVICE) -> np.ndarray:
    """Get embedding for a single sequence."""
    model.eval()
    
    encoded = tokenizer.encode(
        events,
        add_special_tokens=True,
        max_length=MAX_SEQ_LEN,
        padding=True,
        truncation=True
    )
    
    input_ids = torch.tensor([encoded["input_ids"]], device=device)
    attention_mask = torch.tensor([encoded["attention_mask"]], device=device)
    
    embedding = model(input_ids, attention_mask)
    return embedding.cpu().numpy().squeeze()


@torch.no_grad()
def get_embeddings_batch(model, tokenizer, sequences: List[List[str]], device=DEVICE, batch_size=32) -> np.ndarray:
    """Get embeddings for multiple sequences."""
    model.eval()
    
    all_embeddings = []
    
    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i:i + batch_size]
        
        batch_input_ids = []
        batch_attention_mask = []
        
        for seq in batch_seqs:
            encoded = tokenizer.encode(
                seq,
                add_special_tokens=True,
                max_length=MAX_SEQ_LEN,
                padding=True,
                truncation=True
            )
            batch_input_ids.append(encoded["input_ids"])
            batch_attention_mask.append(encoded["attention_mask"])
        
        input_ids = torch.tensor(batch_input_ids, device=device)
        attention_mask = torch.tensor(batch_attention_mask, device=device)
        
        embeddings = model(input_ids, attention_mask)
        all_embeddings.append(embeddings.cpu().numpy())
    
    return np.vstack(all_embeddings)

In [ ]:
# Test embedding generation
sample_sequences = test_seqs[:5]

print("Sample Embeddings:")
print("=" * 60)

for seq in sample_sequences:
    emb = get_embedding(model, tokenizer, seq)
    print(f"\nSequence: {seq[:5]}{'...' if len(seq) > 5 else ''}")
    print(f"Embedding shape: {emb.shape}")
    print(f"Embedding (first 10 dims): {emb[:10].round(4)}")
    print(f"L2 norm: {np.linalg.norm(emb):.4f}")

In [ ]:
# Test similarity between sequences
print("\nSimilarity Test:")
print("=" * 60)

# Get embeddings for test sequences
test_embeddings = get_embeddings_batch(model, tokenizer, test_seqs[:100])

# Compute similarity matrix
sim_matrix = cosine_similarity(test_embeddings)

# Show most similar pairs
print("\nMost similar sequence pairs (excluding self):")
np.fill_diagonal(sim_matrix, -1)  # Exclude self-similarity

for _ in range(3):
    idx = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
    print(f"\nSimilarity: {sim_matrix[idx]:.4f}")
    print(f"  Seq 1: {test_seqs[idx[0]][:6]}")
    print(f"  Seq 2: {test_seqs[idx[1]][:6]}")
    sim_matrix[idx] = -1  # Mark as seen

In [ ]:
# Visualize embeddings with t-SNE
print("\nGenerating t-SNE visualization...")

# Get embeddings for a sample
n_samples = min(500, len(test_seqs))
sample_embeddings = get_embeddings_batch(model, tokenizer, test_seqs[:n_samples])

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(sample_embeddings)

# Plot
plt.figure(figsize=(10, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5, s=10)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title(f't-SNE Visualization of Sequence Embeddings (n={n_samples})')
plt.tight_layout()
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>12. Load and Use Saved Model</b></p>

In [ ]:
def load_embedding_model(model_path: str, device: str = None):
    """
    Load a saved embedding model.
    
    Args:
        model_path: Path to saved model directory
        device: Device to load model to
    
    Returns:
        Tuple of (model, tokenizer, config)
    """
    if device is None:
        device = get_device()
    
    # Load tokenizer
    tokenizer = EventTokenizer.load(model_path)
    
    # Load config
    with open(os.path.join(model_path, "config.json"), "r") as f:
        config = json.load(f)
    
    # Create model
    model = EventSequenceEmbedder(
        vocab_size=config["vocab_size"],
        embedding_dim=config["embedding_dim"],
        d_model=config["d_model"],
        num_heads=config["num_heads"],
        num_layers=config["num_layers"],
        d_ff=config["d_ff"],
        max_seq_len=config["max_seq_len"],
        pad_token_id=config["pad_token_id"],
        pooling=config["pooling"],
        normalize=config["normalize"]
    )
    
    # Load weights
    model.load_state_dict(
        torch.load(os.path.join(model_path, "pytorch_model.bin"), map_location=device)
    )
    model.to(device)
    model.eval()
    
    return model, tokenizer, config


# Example: Load the saved model
print(f"Loading model from: {OUTPUT_DIR}")
loaded_model, loaded_tokenizer, loaded_config = load_embedding_model(OUTPUT_DIR, DEVICE)

print(f"\nLoaded model configuration:")
for k, v in loaded_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Test loaded model
test_seq = test_seqs[0]
print(f"\nTest sequence: {test_seq}")

emb = get_embedding(loaded_model, loaded_tokenizer, test_seq)
print(f"Embedding shape: {emb.shape}")
print(f"Embedding (first 10): {emb[:10].round(4)}")

<hr style="height:1px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>Summary</b></p>
<ol style = 'font-size:16px;font-family:Arial'>In this notebook we created an embedding model for event sequences with:
<li><b>Configurable architecture</b>: tiny/small/medium/large presets or custom  </li>
<li><b>Configurable embedding dimension</b>: Output vector size                  </li>
<li><b>Multiple pooling strategies</b>: mean, max, cls, last                     </li>
<li><b>Contrastive learning</b>: InfoNCE loss for semantic similarity            </li>
<li><b>Data augmentation</b>: For creating positive pairs                        </li>
<li><b>HuggingFace-compatible format</b>: tokenizer.json, config.json, etc.      </li>
</ol>
<p style = 'font-size:18px;font-family:Arial'><b>Output Files</b>
<ol style = 'font-size:16px;font-family:Arial'>embedding_model
<li>pytorch_model.bin      # Model weights             </li>
<li>config.json            # Model architecture config </li>
<li>tokenizer.json         # HuggingFace tokenizer     </li>
<li>vocab.json             # Event vocabulary          </li>
<li>tokenizer_config.json  # Tokenizer settings        </li>
<li>special_tokens_map.json                            </li>
<li>training_config.json   # Training hyperparameters  </li>
<li>README.md              # Model documentation       </li>
</ol>
<p style = 'font-size:18px;font-family:Arial'><b>Use Cases</b>
    <ol style = 'font-size:16px;font-family:Arial'>
<li><b>Vector Store</b>: Index embeddings in Pinecone/Weaviate/Milvus                  </li>
<li><b>Similarity Search</b>: Find similar customer journeys                           </li>
<li><b>Clustering</b>: Group customers by behavior                                     </li>
<li><b>Zero-Shot Classification</b>: Classify sequences by comparing to label examples </li>
<li><b>Anomaly Detection</b>: Flag sequences far from normal patterns                  </li>

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>